# DQN

Même principe que le QLearning.

L'estimation de la fonction Qualité se fait par un réseau au lieu de se faire par une table.

Il est donc tout à fait possible de construire une **table équivalente au réseau**, ce n'est pas la même structure de donnée mais elle représente la même fonction.


Lors de la mise à jour du réseau, de multiples cases de la table équivalente seront modifiées.

Ce phénomène est un **inconvénient** puisqu'en souhaitant apprendre quelque chose nous pouvons détruire ce qui a déjà été appris.

Mais c'est aussi un **avantage** puisqu'il est possible de généraliser notre apprentissage à d'autres situations est donc potentiellement accélérer l'apprentissage.

Le paramétrage de DQN pour RLlib est un peu différent du DQN de StableBaselines mais équivalent

Le paramétrage d'epsilon change, il faut préciser :
- **la valeur de départ d'epsilon** : identique à eps_start
- **la valeur finale d'epsilon** : identique à eps_end
- **le nombre de steps sur lequel epsilon décroit** : assimilé à eps_fraction et temps d'entraînement


DQN partage certains paramètres avec le QLearning :
- **gamma** : identique à gamma

Attention !

Le paramètre **lr** n'est pas identique au alpha du QLearning (même s'il a le même nom)

L'utilisation d'un réseau introduit de nouveaux paramètres :
- **lr** : le pas effectué à chaque rétropropagation du gradient
- **replay_buffer_config : capacity** : la taille du buffer
- **train_batch_size_per_learner** : la taille du batch
- **target_network_update_freq** : l'interval entre deux mise à jour du réseau target
- **model_config : fcnet_hiddens** : l'architecture du réseau (le nombre de couches intermédiaires et le nombre de noeuds par couche)


model_config : fcnet_hiddens = [64, 32] signifie que l'on a 2 couches intermédiaires, la première comporte 64 noeuds et la seconde en a 32.

**Par défaut** :
- **lr** : 0.0005
- **replay_buffer_config : capacity** : 50000
- **train_batch_size_per_learner** : 32
- **gamma** : 0.99
- **target_network_update_freq** : 500
- **nombre de steps sur lequel epsilon décroit** : 10000
- **valeur de départ d'epsilon** : 1.0
- **valeur finale d'epsilon** : 0.02
- **model_config : fcnet_hiddens** : [256, 256]

### **ATTENTION**

La documentation de RLlib n'est pas à jour.

Dans la partie DQN de la documentation, il est noté que training.hiddens permet de choisir l'architecture du réseau, c'est faux.

Il faut bien utiliser model_config : fcnet_hiddens (pour le moment en tout cas)

RLlib change son API et change le nom de certains paramètres, **je ne peux pas garantir qu'à l'heure où vous lisez ceci RLlib fonctionne de la même façon qu'au moment où j'écris ces mots.**


**Afin de vérifier que l'architecture du réseau est correctement mis à jour, n'hésitez pas à print le module obtenu avec algo.get_module()**

**Vous pouvez vérifier le code source si vous le souhaitez (bon courage)**

# DQN de RLlib : Entraînement

## Utilisation basique

RLlib ne permet pas de choisir le nombre de steps sur lesquels l'agent s'entraîne (sans tenter de modifier des paramètres qui auront diverses conséquences)

In [ ]:
import ray
from ray import train
from ray.rllib.algorithms.dqn import DQNConfig
from ray.rllib.connectors.env_to_module import FlattenObservations


# Choisir le nombre minimal de steps sur lequel s'entraîner
timesteps = 6000


config = (
    DQNConfig()
    .environment(
        MonEnvironnement,  # On renseigne la classe
        env_config = {}  # C'est le dictionnaire options
    )

    # Si votre environnement utilise un spaces.Discrete ou MultiDiscrete, écrivez ces lignes
    # Afin que RLlib fasse un One Hot Encoding, sinon cela ne fonctionnera pas
    .env_runners(
        env_to_module_connector = lambda env: FlattenObservations()
    )
)



algo = config.build_algo()


timesteps_done = 0

while timesteps_done < timesteps :
    # On entraîne pour un certain nombre de steps inconnu à l'avance
    results = algo.train()

    # On récupère le nombre de steps effectués en tout
    timesteps_done = results['env_runners']['num_env_steps_sampled_lifetime']



algo.stop()
ray.shutdown()

## Obtenir le gain par épisode

Pour obtenir le gain par épisode, on utilise le dictionnaire results qui est donné à chaque appel à la méthode train().

Ce grand dictionnaire regroupe des informations qui permettent l'affichage du gain moyen au cours des épisodes effectués lors du dernier appel à train()

In [ ]:
import ray
from ray import train
from ray.rllib.algorithms.dqn import DQNConfig
from ray.rllib.connectors.env_to_module import FlattenObservations
import matplotlib.pyplot as plt

timesteps = 6000


config = (
    DQNConfig()
    .environment(
        MonEnvironnement,
        env_config = {}
    )

    .env_runners(
        env_to_module_connector = lambda env: FlattenObservations()
    )
)



algo = config.build_algo()


ordonnees = []
toPlot = []
episode_count = 0
timesteps_done = 0

while timesteps_done < timesteps :
    # On entraîne pour un certain nombre de steps
    results = algo.train()

    # On récupère le nombre de steps effectués sur l'ensemble de l'entraînement
    timesteps_done = results['env_runners']['num_env_steps_sampled_lifetime']

    # On ajoute le nombre d'épisodes effectués au cours du dernier appel à train()
    episode_count += results['env_runners']['num_episodes']

    # On récupère le gain moyen au cours des épisodes effectués pendant le dernier appel à train()
    toPlot.append(results['env_runners']['episode_return_mean'])
    ordonnees.append(episode_count)

algo.stop()


plt.plot(ordonnees, toPlot) # Attention à bien plot sur la bonne échelle à l'aide de "ordonnees"
plt.show()
ray.shutdown()

## Paramétrer DQN

Voyons comment spécifier les paramètres de DQN

Les paramètres sont à préciser dans la partie .training() sauf pour fcnet_hiddens.

Il y a aussi une syntaxe particulière pour epsilon.

De plus, RLlib est fait pour utiliser du calcul parallèle.

In [ ]:
config = (
    DQNConfig()
    .environment(
        MonEnvironnement,
        env_config = {}  # Dictionnaire options
    )
    
    .env_runners(
        # One Hot Encoding
        env_to_module_connector=lambda env: FlattenObservations(),
        
        # Pour le calcul parallèle
        num_env_runners=3,
        num_envs_per_env_runner = 10
    )


    .training(
        # Quelques paramètres classiques
        lr = 0.001,
        gamma = 0.9,

        # Pour epsilon on utilise : [[0, eps_start], [nbStepsPourDécroissance, eps_end]]
        epsilon = [[0, 1], [80000, 0.01]],  # Ici epsilon commence à 1 et décroit linéairement jusqu'à
        # atteindre 0.01 au step 80000
    )

    # L'architecture des réseaux se configure de cette façon.
    .rl_module(
        model_config = {
            "fcnet_hiddens" : [32, 32]  # 2 couches intermédiaires de 32 noeuds
        }
    )
)

ray.shutdown()

## Faire un gridSearch avec tune

RLlib dispose d'un outil pour accélérer le gridSearch.

Le gridSearch consiste à préciser quels valeurs on souhaite essayer pour chaque paramètre, puis on essaye toutes les combinaisons possibles parmis les choix effectués.

In [ ]:
config = (
    DQNConfig()
    .environment(
        LivraisonRLlibEnv,
        env_config = {'carte' : generate_random_map(10)},
    )
    
    .env_runners(
        env_to_module_connector=lambda env: FlattenObservations(),
        num_env_runners=3,
        num_envs_per_env_runner = 10
    )


    .training(
        lr = 0.001,
        gamma = 0.9,
        epsilon = [[0, 1], [80000, 0.01]],


        # tune.grid_search pour préciser les valeurs possibles pour ces paramètres
        train_batch_size_per_learner=tune.grid_search([512, 1024, 2048, 4096]),
        target_network_update_freq = tune.grid_search([100, 500, 750, 1000])
    )

    .rl_module(
        model_config = {
            "fcnet_hiddens" : [32, 32]
        }
    )
)


# Une fois la config terminée, on utilise tune.Tuner()

tuner = tune.Tuner(
    config.algo_class,  # La classe de l'algorithme dont nous avons créé la configuration
    param_space=config, # La configuration que nous avons créé pour cet algorithme
    run_config=train.RunConfig(

        # Condition d'arrêt d'entraînement pour chaque essai.
        # Ici, on arrête chaque entraînement au bout de 100000 steps effectués
        # Il est possible de définir des conditions sur le gain moyen
        stop={"num_env_steps_sampled_lifetime": 100000},
    ),
)
results = tuner.fit()  # Effectue l'entraînement pour chaque combinaison


ray.shutdown()

Les résultats de ces entraînements peuvent être récupérés dans des fichiers CSV.